# Practical 8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearnex import patch_sklearn
patch_sklearn(verbose=False)

## Exercise 1: Introduction to simple text embeddings

In this exercise you will learn basic mathods to deal with text data. We will use with the famous `20newsgroups` dataset. It consists of a large collection of news posts across 20 topics. We will be using it to test some basic NLP techniques and train a multi-class classification model to predict the most likely topic for unseen news posts. 
For more information, check [the dataset description](https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset) and the [import function helper](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html)

### 1.0 Importing text data

We restrict the dataset to only 4 of the categories, for presentation simplicity.

In [ ]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# Import training data
twenty_train = fetch_20newsgroups(subset='train',
    categories=categories, shuffle=True, random_state=42)

# Import test data
twenty_test = fetch_20newsgroups(subset='test',
    categories=categories, shuffle=True, random_state=42)
docs_test = twenty_test.data

Try to access from `twent_train` the categories you want to predict.

Try to access the first observation from the training data `twenty_train`. Consider using the `print()` function.

### 1.1 Text vectorization example with Bag of words and TF-IDF

We want to train a simple multi-class ML classifier to predict the news topic. In order to do so, we must first vectorize the text data.

Here we perform basic text embedding using bag-of-words and TF-IDF.

Bag-of-words is the simplest *embedding* technique in which words are represented as one-hot encoded numeric vectors of word counts. The length of these vectors corresponds to the size of the vocabulary of the training corpus.
That is, each column $j$ in the resulting sparse matrix represents a word and and each row $i$ represents a different observation (i.e. document), and the coresonping entry in the matrix is the word count of how many times the word $j$ appears in document $i$.
See [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) for the scikit-learn implementation of the bag-of-words embedding and its many options.

TF-IDF uses the **same** one-hot encoding as traditional BOW, but transforms the simple counts to the relative word frequency, normalized by the inverse-document-frequency to account for frequently occurring words across all documents. 
The intuition is that not only the word's frequency in a given document indicates if that word represents the document well, but also how rare it is in other documents in comparison. 
See the [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) and [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html) scikit-learn classes for help.

For more information see
- https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

- https://en.wikipedia.org/wiki/Tf-idf

#### Way 1: Bag of words and TF-IDF separately

Here we perform the preprocessing in two steps. 
First, we create the predictor matrix `X_train_counts` using bag of words (`CountVectorizer`). Each column here represents a word and and each row represents a different observation, i.e., document. `CountVectorizer` counts how many times the word $j$ appears in document $i$. 

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
#Some arguments: max_features=1500, min_df=5, max_df=0.7, ngram_range=(1,2), stop_words=nltk.stopwords.words('english')
X_train_counts = count_vect.??


How many documents are there in the data sets? How many words?

Now create the predictor matrix `X_train_tfidf` using `TfidfTransformer`. What is the difference between `X_train_counts` and `X_train_tfidf`?

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.??

#### Way 2: both at once
Here we repeat the two steps above in one single step, using `TfidfVectorizer`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidfconverter = TfidfVectorizer()
X_train_tfidf = tfidfconverter.??

### 1.2 Text classification

Fit a logistic regression model on the data set. Evaluate the test accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.??

Repeat the steps of preprocessing and fitting the model with a `Pipeline`.

### 1.3 Tuning the hyper-parameters

One can now perform a cross-validated grid search to select the best hyper-parameter values for both the `TfidfVectorizer` and `LogisticRegression` model at the same time using a sklearn `Pipeline`. For the TF-IDF Vectorizer, let's for example check if considering bigrams in the vocabulary helps the classifier. For the logistic regression, the tuning parameter is the cost ($1/penalty$) `'C'`.
What are the roles of the hyperparameters?

In [ ]:
from sklearn.model_selection import KFold, GridSearchCV

In [ ]:
# Define folds
folds = KFold(n_splits=5, shuffle=True, random_state=42)

# Define parameter grid
my_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    "logistic__C": [10, 100, 1000],
}

# Define grid search CV
??

In [ ]:
# Run CV
??

Compute the test accuracy of the best model obtained with the cross-validation above. Compare it with the baseline model.

### 1.4 Model diagnostics
We can run some further classification accuracy diagnostics to understand in more detail how our model is performing. Consider using the `confusion_matrix()` and `classification_report()` functions from the `sklearn` library.

### 1.5 Predict some sentences
Type some sentences and check if they are predicted correctly.

In [ ]:
docs_new = [??, ??, ??]

predicted = best_model.predict(docs_new)

for doc, category in zip(docs_new, predicted):
    print(doc, "=>", twenty_train.target_names[category])

## To go further...

Other more advanced word embedding methods can be found in recent literature, and some are available in other packages, or online in trainable or pretrained versions. 
Those include static embeddings: (see word2vec, GloVe, fastText) and context dynamic embeddings: (Transformers, BERT, GPT, ...).

A few python libraries I can recommend you look up if you're interested: `nltk`, `gensim`, the huggingface libraries, ...

## Appendix: Text wrangling and preprocessing

In practice, textual datasets are rarely clean nor well structured, and often need some wrangling and preprocessing to be used effectively. 

Furthermore, depending on the specific task and context at hand, there are often other tailor-made transformations that can prove usefull as an addition or a replacement to normalization. (E.g. the way you would like to handle the `@`symbol might differ between e-mail and social media data. Or there might be specific groups of words that have similar meaning in general, but whose differentiation is important in a specific context.)

Additionally to vectorisation, manual feature extraction can also prove useful. For example the number of exclamation marks, the number of ALL CAPS WORDS, or the average word per sentence ratio might give additional information on the tone or sentiment of written text, depending on the context and model.

Here are a few basic string methods that can come in handy for those scenarios.

In [ ]:
import string

In [ ]:
text = "Hi there!"
text

In [ ]:
text.replace("Hi","Hello")

In [ ]:
text.replace("!"," ! ")

In [ ]:
text.replace("e","")

In [ ]:
text.split(" ")

In [ ]:
text.lower()

In [ ]:
string.punctuation

The few examples above are just to inspire you some ideas. There are many things you could think of to analyze and extract informative summaries from text data. The pandas `<pd.Series>.apply()` method can come in very handy with custom user-defined functions.

pd.Series also has a `str` subset of functions for text data. Here are a few dummy examples.

In [ ]:
str_text = pd.DataFrame({"my_text":["Hi there!","My dog is cute.","i lost my wallet"]})
str_text

In [ ]:
str_text.my_text.str.capitalize()

In [ ]:
str_text.my_text.str.lower()

In [ ]:
str_text.my_text.str.contains("y")

In [ ]:
str_text.my_text.str.contains("my")

In [ ]:
str_text.my_text.str.count("e")

In [ ]:
str_text.my_text.str.replace("e","")

Many more examples in the pandas documentation.

For more complicated text processing procedures, one would usually turn to [**regular expressions**](https://en.wikipedia.org/wiki/Regular_expression), as a much more powerful tool. The [`re` module](https://docs.python.org/3/library/re.html) provides the base tools to work with regular expressions in python. Some `pandas`'s  `Series.str` methods above also accept regular expressions.

This goes beyond the scope of this seminar, but if you are interested:
- [Interactive RegEx tutorials](https://regexr.com/)
- [Another tutorial](https://www.w3schools.com/python/python_regex.asp)
- And many more...